In [8]:
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from multi_agent_system.config import settings

from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

load_dotenv()

True

In [ ]:

# Verzeichnis, in dem die Markdown-Dateien liegen
DOCUMENT_DIR = Path(r"C:\Programming\multi-agent-system\data\extracted_markdown").resolve()


class DocumentClassification(BaseModel):
    document_type: Literal["Rechnung", "Angebot"] = Field(description="Klassifikation des Dokuments")
    confidence: float = Field(ge=0.0,le=1.0,description="Sicherheit der Klassifikation zwischen 0 und 1")
    reason: str = Field(description="Kurze Begründung für die Klassifikation")


@tool
def read_markdown_file(name: str) -> str:
    """
    Reads a Markdown document from the documents directory.

    Use this tool whenever a Markdown file needs to be classified.
    The filename must end with .md.
    """

    if not name.endswith(".md"):
        return "ERROR: Only Markdown files (.md) are supported."

    file_path = (DOCUMENT_DIR / name).resolve()

    # Verhindert Zugriff außerhalb von DOCUMENT_DIR
    if DOCUMENT_DIR not in file_path.parents:
        return "ERROR: Invalid file path."

    if not file_path.exists():
        return f"ERROR: File '{name}' was not found."

    if not file_path.is_file():
        return f"ERROR: '{name}' is not a file."

    try:
        return file_path.read_text(encoding="utf-8")
    except Exception as exc:
        return f"ERROR: Could not read file: {exc}"


model = init_chat_model(
    "gpt-4o-mini",
    openai_api_key=settings.openai_api_key.get_secret_value(),
    temperature=0,
)


agent = create_agent(
    model=model,
    tools=[read_markdown_file],
    response_format=DocumentClassification,
    system_prompt="""
        You are a document classification agent.

        Your task is to classify Markdown documents as either:

        - Rechnung
        - Angebot

        Always use the read_markdown_file tool to read the requested document
        before making a classification.

        Classification rules:

        Rechnung:
        - requests payment for goods or services already delivered
        - often contains terms such as Rechnungsnummer, Rechnungsdatum,
        Zahlungsziel, fälliger Betrag, Bankverbindung or Zahlung
        - usually contains a final amount that must be paid

        Angebot:
        - proposes goods or services that can be purchased
        - often contains terms such as Angebot, Angebotsnummer,
        Gültigkeit, Leistungsbeschreibung or Angebotspreis
        - does not represent an immediate payment obligation

        Base the classification on the semantic meaning of the whole document,
        not only individual keywords.

        Return a short reason and a confidence score.
        """
)


result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Checke die Markdown file ob es sich um eine Rechnung oder ein Angebot handelt. "
            }
        ]
    }
)

classification = result["structured_response"]

print(classification.document_type)
print(classification.confidence)
print(classification.reason)


Angebot
0.8
Das Dokument enthält Begriffe wie 'Angebot' und 'Angebotspreis', was darauf hindeutet, dass es sich um ein Angebot handelt, das keine sofortige Zahlungspflicht darstellt.


In [ ]:
import json
from pathlib import Path
from typing import Literal
from enum import Enum
from multi_agent_system.config import settings
from dotenv import load_dotenv

from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

load_dotenv()

DOCUMENT_DIR = Path(
    r"C:\Programming\multi-agent-system\data\extracted_markdown"
).resolve()

OUTPUT_DIR = Path(
    r"C:\Programming\multi-agent-system\data\extracted_json"
).resolve()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------------
# Enums
# ------------------------------------------------------------------

class Einheit(str, Enum):
    STUECK = "Stück"
    METER = "m"
    QUADRATMETER = "m²"
    STUNDE = "Stunde"


# ------------------------------------------------------------------
# Datenmodelle
# ------------------------------------------------------------------

class Kunde(BaseModel):
    firmen_name: str | None = None
    strasse: str | None = None
    postleitzahl: str | None = None
    stadt: str | None = None
    telefon: str | None = None
    email: str | None = None


class Position(BaseModel):
    position: int
    beschreibung: str

    menge: float | None = Field(
        default=None,
        description="Menge der Position"
    )

    einheit: Einheit | None = Field(
        default=None,
        description="Einheit der Position"
    )

    preis: float | None = Field(
        default=None,
        description="Der Gesamtpreis der Position"
    )


class DokumentDaten(BaseModel):
    dokumententyp: Literal["angebot", "rechnung"]
    angebotsnummer: str | None = None
    rechnungsnummer: str | None = None
    kunde: Kunde
    positionen: list[Position]


# ------------------------------------------------------------------
# Tool
# ------------------------------------------------------------------

@tool
def read_markdown_file(name: str) -> str:
    """
    Reads the content of a Markdown document.
    """

    if not name.lower().endswith(".md"):
        return "ERROR: Only Markdown files (.md) are supported."

    file_path = (DOCUMENT_DIR / name).resolve()

    if DOCUMENT_DIR not in file_path.parents:
        return "ERROR: Invalid file path."

    if not file_path.exists():
        return f"ERROR: File '{name}' was not found."

    if not file_path.is_file():
        return f"ERROR: '{name}' is not a file."

    try:
        return file_path.read_text(encoding="utf-8")
    except Exception as exc:
        return f"ERROR: Could not read file: {exc}"


# ------------------------------------------------------------------
# Model
# ------------------------------------------------------------------

model = init_chat_model(
    "gpt-5-nano",
    openai_api_key=settings.openai_api_key.get_secret_value(),
    temperature=0,
)


# ------------------------------------------------------------------
# Agent
# ------------------------------------------------------------------

agent = create_agent(
    model=model,
    tools=[read_markdown_file],
    response_format=DokumentDaten,
    system_prompt="""
You are an agent for processing German business documents.

Your job is to:

1. Read the requested Markdown file using the read_markdown_file tool.
2. Determine whether the document is:
   - "angebot"
   - "rechnung"
3. Extract the structured information from the document.

IMPORTANT RULES:

DOCUMENT TYPE
-------------
Use "angebot" if the document proposes products or services
that may be purchased.

Use "rechnung" if the document requests payment for products
or services that have already been delivered or performed.


DOCUMENT NUMBER
---------------
If the document is an Angebot:
- set dokumententyp to "angebot"
- extract angebotsnummer
- set rechnungsnummer to null

If the document is a Rechnung:
- set dokumententyp to "rechnung"
- extract rechnungsnummer
- set angebotsnummer to null


CUSTOMER
--------
"kunde" always refers to the CUSTOMER / RECIPIENT of the document.

Do NOT accidentally extract the company that created or sent
the invoice or offer as the customer.

Extract when available:

- firmen_name
- strasse
- postleitzahl
- stadt
- telefon
- email

If a value is not present in the document, return null.
Never invent missing information.

POSITIONS
---------
Extract all invoice or offer positions.

For each position extract:

- position
- beschreibung
- menge
- einheit
- preis

Do use preis as the total price of the position, not the unit price.
Preis should result in:
{
  "menge": 5,
  "einheit": "Stück",
  "preis": means "Gesamt", "Gesamtpreis", "Gesamtbetrag",
}

Allowed values for "einheit":
- Stück
- m
- m²
- Stunde

Normalize common variants:

Stk., Stk -> Stück
Meter, lfm, Laufmeter -> m
qm, m2 -> m²
Std., Stunden -> Stunde


Preserve the order of the positions from the document.

Do not invent information that is not contained in the document.
"""
)


# ------------------------------------------------------------------
# Verarbeitung
# ------------------------------------------------------------------

def process_document(filename: str) -> Path:

    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        f"Lies die Datei '{filename}', "
                        "klassifiziere sie und extrahiere die Daten."
                    )
                }
            ]
        }
    )

    document: DokumentDaten = result["structured_response"]

    # Wichtig: Enums werden zu JSON-Werten konvertiert
    data = document.model_dump(
        mode="json",
        exclude_none=True
    )

    output_file = OUTPUT_DIR / f"{Path(filename).stem}.json"

    with output_file.open("w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2
        )

    return output_file


output_file = process_document("angebot_test_output.md")

print(f"JSON gespeichert unter: {output_file}")

JSON gespeichert unter: C:\Programming\multi-agent-system\data\extracted_json\angebot_test_output.json
